In [3]:
!pip install captum lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 18.3 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=34985c2c8eac0ce2a0aaa9ae375ae0ac3e2455b914bd2a33b1c430fb70a3ba4c
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [4]:
import os
import time
import csv
import psutil
import zipfile
import shutil
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import swin_t, Swin_T_Weights

from captum.attr import IntegratedGradients
from lime import lime_image
from scipy.integrate import simpson

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
class CustomMedMNISTDataset(Dataset):
    def __init__(self, npz_path, split='train', transform=None, extract_dir='/content/tmp_mmap'):
        self.transform = transform
        os.makedirs(extract_dir, exist_ok=True)

        base_name = os.path.basename(npz_path).replace('.npz', '')
        self.img_path = os.path.join(extract_dir, f"{base_name}_{split}_images.npy")
        self.lbl_path = os.path.join(extract_dir, f"{base_name}_{split}_labels.npy")

        if not (os.path.exists(self.img_path) and os.path.exists(self.lbl_path)):
            print(f"Streaming {split} arrays to disk for {base_name}...")
            with zipfile.ZipFile(npz_path, 'r') as zip_ref:
                with zip_ref.open(f'{split}_images.npy') as src, open(self.img_path, 'wb') as tgt:
                    shutil.copyfileobj(src, tgt)
                with zip_ref.open(f'{split}_labels.npy') as src, open(self.lbl_path, 'wb') as tgt:
                    shutil.copyfileobj(src, tgt)

        self.images = np.load(self.img_path, mmap_mode='r')
        self.labels = np.load(self.lbl_path, mmap_mode='r')
        self.is_grayscale = (len(self.images.shape) == 3)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        labels = self.labels[idx].astype(np.float32)

        img = Image.fromarray(img_array, mode='L' if self.is_grayscale else 'RGB')
        if self.transform:
            img = self.transform(img)

        return img, labels

def prepare_dataloaders(npz_file_name, data_dir, batch_size=64, num_bench_samples=100):
    npz_path = os.path.join(data_dir, npz_file_name)
    data_transform = transforms.Compose([
        transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(CustomMedMNISTDataset(npz_path, 'train', data_transform), batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(CustomMedMNISTDataset(npz_path, 'val', data_transform), batch_size=batch_size, shuffle=False, num_workers=2)

    test_dataset = CustomMedMNISTDataset(npz_path, 'test', data_transform)
    bench_loader = DataLoader(Subset(test_dataset, list(range(num_bench_samples))), batch_size=1, shuffle=False)

    return train_loader, val_loader, bench_loader